Tables we may recover

In [0]:
RECOVERY_TABLES = {
    "bronze_customers":
        "customer360_dev.bronze.customers",

    "silver_customers":
        "customer360_dev.silver.customers",

    "gold_dim_customer":
        "customer360_dev.gold.dim_customer",

    "gold_customer360":
        "customer360_dev.gold.agg_customer_360"
}


Inspect Delta history

In [0]:
for name, table_name in RECOVERY_TABLES.items():

    print("=" * 80)
    print(name, "->", table_name)

    display(
        spark.sql(
            f"DESCRIBE HISTORY {table_name}"
        )
        .select(
            "version",
            "timestamp",
            "operation",
            "operationParameters",
            "userName"
        )
        .orderBy("version", ascending=False)
        .limit(20)
    )

bronze_customers -> customer360_dev.bronze.customers


version,timestamp,operation,operationParameters,userName
3,2026-09-27T13:10:19.000Z,COMPUTE STATS,"Map(predicate -> [""true""], incremental -> false, buildFromFooter -> false)",56b0abdf-3cbb-4bc5-825d-3271bff34e2b
2,2026-09-27T13:10:03.000Z,SET TBLPROPERTIES,"Map(properties -> {""delta.workloadBasedColumns.deltaFileStatistics"":""`customer_id`""})",56b0abdf-3cbb-4bc5-825d-3271bff34e2b
1,2026-09-27T07:01:23.000Z,ADD COLUMNS,"Map(columns -> [{""column"":{""name"":""_pipeline_run_id"",""type"":""string"",""nullable"":true,""metadata"":{}}}])",pavankumar.m1432@gmail.com
0,2026-09-26T09:46:14.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""2c15c167-0e89-4142-b367-3e977ca36ecd""}, statsOnLoad -> true)",pavankumar.m1432@gmail.com


silver_customers -> customer360_dev.silver.customers


version,timestamp,operation,operationParameters,userName
3,2026-09-27T12:35:35.000Z,COMPUTE STATS,"Map(predicate -> [""true""], incremental -> false, buildFromFooter -> false)",56b0abdf-3cbb-4bc5-825d-3271bff34e2b
2,2026-09-27T12:35:23.000Z,SET TBLPROPERTIES,"Map(properties -> {""delta.workloadBasedColumns.deltaFileStatistics"":""`customer_id`""})",56b0abdf-3cbb-4bc5-825d-3271bff34e2b
1,2026-09-27T06:47:15.000Z,MERGE,"Map(predicate -> [""(customer_id#31252 = customer_id#31049)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""((cdc_action#31047 = UPDATE) AND (isnull(updated_at#31264) OR (updated_at#31061 >= updated_at#31264)))"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""predicate"":""(cdc_action#31047 = INSERT)"",""actionType"":""insert""}])",pavankumar.m1432@gmail.com
0,2026-09-26T10:35:27.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""957f377e-0637-45b9-bf07-792d8204046c""}, statsOnLoad -> true)",pavankumar.m1432@gmail.com


gold_dim_customer -> customer360_dev.gold.dim_customer


version,timestamp,operation,operationParameters,userName
2,2026-09-29T23:20:32.000Z,MERGE,"Map(predicate -> [""((customer_id#20165 = customer_id#20121) AND is_current#20178)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",pavankumar.m1432@gmail.com
1,2026-09-29T23:15:07.000Z,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",pavankumar.m1432@gmail.com
0,2026-09-29T23:10:31.000Z,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""99449570-efe6-4ed8-8295-e3e286ff4964"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a038cf27-18b2-4c0b-905c-c0bec9f66313"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-59b83fce-7a4e-46dd-ba6e-a9dacfdbe8a2""}, statsOnLoad -> false)",pavankumar.m1432@gmail.com


gold_customer360 -> customer360_dev.gold.agg_customer_360


version,timestamp,operation,operationParameters,userName
0,2026-09-29T23:43:39.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""0c665afd-6b99-44b5-aae3-69ce36daef53""}, statsOnLoad -> true)",pavankumar.m1432@gmail.com


Inspect a previous version before restoring

In [0]:
table_name = "customer360_dev.silver.customers"

previous_version = 5

In [0]:
# Determine the latest available Delta version for time travel
previous_version = (
    spark.sql(f"DESCRIBE HISTORY {table_name}")
    .select("version")
    .orderBy("version", ascending=False)
    .first()["version"]
)

previous_df = (
    spark.read
    .format("delta")
    .option(
        "versionAsOf",
        previous_version
    )
    .table(table_name)
)

print(
    "Rows in previous version:",
    previous_df.count()
)

Rows in previous version: 986000


Inspect a previous version before restoring

In [0]:
current_df = spark.table(
    table_name
)

current_count = current_df.count()
previous_count = previous_df.count()

print(
    "Current count :",
    current_count
)

print(
    "Previous count:",
    previous_count
)

print(
    "Difference    :",
    current_count - previous_count
)

Current count : 986000
Previous count: 986000
Difference    : 0


In [0]:
missing_from_current = (
    previous_df
    .select("customer_id")
    .join(
        current_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Customers present before but missing now:",
    missing_from_current.count()
)

Customers present before but missing now: 0


Restore table to previous version

In [0]:
%sql
RESTORE TABLE customer360_dev.silver.customers
TO VERSION AS OF 3

In [0]:
restore_version = 3

spark.sql(
    f"""
    RESTORE TABLE
        customer360_dev.silver.customers
    TO VERSION AS OF
        {restore_version}
    """
)

print(
    "Silver customers restored to version:",
    restore_version
)

Silver customers restored to version: 3


In [0]:
restored_df = spark.table(
    "customer360_dev.silver.customers"
)

print(
    "Restored Silver customer count:",
    restored_df.count()
)

Restored Silver customer count: 986000


In [0]:
display(
    spark.sql(
        """
        DESCRIBE HISTORY
        customer360_dev.silver.customers
        """
    )
    .select(
        "version",
        "timestamp",
        "operation"
    )
    .orderBy(
        "version",
        ascending=False
    )
    .limit(10)
)

version,timestamp,operation
3,2026-09-27T12:35:35.000Z,COMPUTE STATS
2,2026-09-27T12:35:23.000Z,SET TBLPROPERTIES
1,2026-09-27T06:47:15.000Z,MERGE
0,2026-09-26T10:35:27.000Z,CREATE OR REPLACE TABLE AS SELECT


8. Example: rebuild Silver from Bronze
For a corrupted Silver table, the preferred recovery can be:

Bronze raw history
      ↓
reapply deterministic Silver rules
      ↓
replace corrected Silver

In [0]:
bronze_customer_df = spark.table(
    "customer360_dev.bronze.customers"
)

# Reuse the same transformation rules from 09_silver_customers.
# Do not invent different logic during recovery.

Example: rebuild Gold Customer 360
If only Gold is damaged, we do not touch Bronze or Silver.

In [0]:
# Re-run the Gold build logic from:
# 48_build_gold_customer360

print(
    "Rebuild Gold only from trusted Silver."
)

Rebuild Gold only from trusted Silver.


In [0]:
%sql
SELECT
    entity_name,
    last_processed_value,
    last_processed_file,
    last_run_id,
    load_status,
    updated_at
FROM customer360_dev.control.watermark
ORDER BY entity_name;

entity_name,last_processed_value,last_processed_file,last_run_id,load_status,updated_at
addresses,null,null,null,SUCCESS,2026-09-27T11:59:30.558Z
customers,null,null,null,SUCCESS,2026-09-27T11:27:19.887Z
order_items,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
orders,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
payments,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
products,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z
subscriptions,null,null,null,SUCCESS,2026-09-29T22:56:03.608Z
support_tickets,null,null,null,INITIAL_BASELINE_COMPLETE,2026-09-27T06:04:23.147Z


In [0]:
from pyspark.sql import functions as F
checks = {}

checks["silver_customer_duplicates"] = (
    spark.table(
        "customer360_dev.silver.customers"
    )
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks["gold_current_customer_duplicates"] = (
    spark.table(
        "customer360_dev.gold.dim_customer"
    )
    .filter(
        F.col("is_current") == True
    )
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks["customer360_duplicates"] = (
    spark.table(
        "customer360_dev.gold.agg_customer_360"
    )
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

for check_name, value in checks.items():
    print(
        check_name,
        "=",
        value
    )

recovery_status = (
    "PASS"
    if all(
        value == 0
        for value in checks.values()
    )
    else "FAIL"
)

print(
    "Recovery verification:",
    recovery_status
)

silver_customer_duplicates = 0
gold_current_customer_duplicates = 0
customer360_duplicates = 0
Recovery verification: PASS
